# Adaptive Layer Selection (Early Exit)

## Learning Objectives
1. Understand confidence-based early exit for inference acceleration
2. Implement a multi-exit MLP with shared backbone
3. Analyze exit-layer distribution vs threshold and accuracy
4. Compare static vs dynamic depth strategies

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import List, Tuple, Optional

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## Level 1: Basic Early Exit Simulation

In [ ]:
np.random.seed(42)
n_layers, n_samples, n_classes = 12, 300, 5

# Each sample has a difficulty in [0,1] — easy samples converge faster
difficulties = np.random.beta(2, 2, n_samples)  # bell-shaped around 0.5

def layer_confidence(layer_idx, difficulty, noise_std=0.04):
    """Confidence increases with depth; harder samples need more layers."""
    base = (layer_idx + 1) / n_layers
    noise = np.random.normal(0, noise_std)
    return float(np.clip(base * (2 - difficulty) + noise, 0, 1))

thresholds = [0.50, 0.65, 0.80, 0.92]

print(f"{'Threshold':<12} {'Avg layers':<13} {'Speedup':<10} {'Approx acc'}")
print("-" * 48)
for threshold in thresholds:
    exit_layers, accs = [], []
    for s in range(n_samples):
        exited = False
        for l in range(n_layers):
            conf = layer_confidence(l, difficulties[s])
            if conf >= threshold:
                exit_layers.append(l + 1)
                # Accuracy proxy: harder samples that exit early are more likely wrong
                acc = 1.0 if l + 1 >= difficulties[s] * n_layers else 0.65
                accs.append(acc)
                exited = True
                break
        if not exited:
            exit_layers.append(n_layers)
            accs.append(1.0)
    avg_l  = np.mean(exit_layers)
    approx = np.mean(accs)
    speedup = n_layers / avg_l
    print(f"{threshold:<12.2f} {avg_l:<13.2f} {speedup:<10.2f} {approx:.3f}")

# Exit distribution for threshold=0.80
exits_80 = []
for s in range(n_samples):
    for l in range(n_layers):
        if layer_confidence(l, difficulties[s]) >= 0.80:
            exits_80.append(l + 1)
            break
    else:
        exits_80.append(n_layers)

print(f"\nExit distribution (threshold=0.80):")
for l in range(1, n_layers + 1):
    count = exits_80.count(l)
    bar = '|' * count
    print(f"  Layer {l:2d}: {count:3d} samples  {bar}")

## Level 2: Multi-Exit MLP with PyTorch

In [ ]:
class MultiExitMLP(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, n_classes=5,
                 n_layers=12, exit_at=(4, 8, 12)):
        super().__init__()
        self.exit_at = list(exit_at)
        self.layers  = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim if i == 0 else hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ReLU()
            ) for i in range(n_layers)
        ])
        self.exit_heads = nn.ModuleDict({
            str(l): nn.Linear(hidden_dim, n_classes) for l in exit_at
        })
        self.n_layers = n_layers

    def forward(self, x: torch.Tensor, threshold: float = 0.85):
        h = x
        for idx, layer in enumerate(self.layers):
            h = layer(h)
            l = idx + 1
            if l in self.exit_at:
                logits = self.exit_heads[str(l)](h)
                probs  = torch.softmax(logits, dim=-1)
                conf   = probs.max(dim=-1).values  # [B]
                if not self.training and (conf >= threshold).all():
                    return logits, l, conf.mean().item()
        # Guaranteed final exit
        logits = self.exit_heads[str(self.exit_at[-1])](h)
        probs  = torch.softmax(logits, dim=-1)
        return logits, self.exit_at[-1], probs.max(dim=-1).values.mean().item()

model = MultiExitMLP().to(device)
print("Model parameter count:", sum(p.numel() for p in model.parameters()))

# Simulate inference with different thresholds
B, dim = 32, 64
x = torch.randn(B, dim, device=device)

print(f"\n{'Threshold':<12} {'Exit layer':<13} {'Avg conf':<12} {'Layer FLOPs saved'}")
print("-" * 52)
model.eval()
with torch.no_grad():
    for th in [0.50, 0.70, 0.85, 0.95, 1.01]:
        logits, exit_l, avg_conf = model(x, threshold=th)
        pred = logits.argmax(-1)
        layers_saved = 12 - exit_l
        print(f"{th:<12.2f} {exit_l:<13d} {avg_conf:<12.3f} {layers_saved}/12")

# Training loop (brief demo)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
model.train()
losses = []
for step in range(200):
    x   = torch.randn(B, dim, device=device)
    y   = torch.randint(0, 5, (B,), device=device)
    # Sum loss over all exit heads for joint training
    total_loss = torch.tensor(0.0, device=device)
    h = x
    for idx, layer in enumerate(model.layers):
        h = layer(h)
        l = idx + 1
        if l in model.exit_at:
            logits = model.exit_heads[str(l)](h)
            total_loss = total_loss + nn.functional.cross_entropy(logits, y)
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    losses.append(total_loss.item())
print(f"\nTraining loss: {losses[0]:.3f} -> {losses[-1]:.3f}")
# ─── Extended calibration study ─────────────────────────────────────
print("\n=== Exit Layer Distribution Study ===")
np.random.seed(0)
n_s2, n_l2, exit_pts2 = 1000, 12, [3, 6, 9, 12]

def sim_exit(difficulty, threshold, exit_pts):
    for l in exit_pts:
        conf = float(np.clip((l/n_l2)*(1.5 - difficulty) + np.random.normal(0, 0.05), 0, 1))
        if conf >= threshold or l == exit_pts[-1]:
            return l
    return exit_pts[-1]

diffs2 = np.random.beta(1.5, 1.5, n_s2)
print(f"{'Threshold':<12} {'L3 exits':<10} {'L6 exits':<10} {'L9 exits':<10} {'L12 exits':<10} {'Speedup'}")
print("-" * 62)
for th in [0.55, 0.65, 0.75, 0.85, 0.95]:
    exits = [sim_exit(d, th, exit_pts2) for d in diffs2]
    counts = {l: exits.count(l) for l in exit_pts2}
    avg_l  = np.mean(exits)
    speedup = n_l2 / avg_l
    row = '  '.join(f"{counts[l]:6d}" for l in exit_pts2)
    print(f"{th:<12.2f} {row}  {speedup:.2f}x")

# Accuracy vs threshold trade-off
print("\n=== Accuracy-Speedup Pareto Frontier ===")
for th in np.arange(0.5, 1.05, 0.05):
    exits = [sim_exit(d, th, exit_pts2) for d in diffs2]
    avg_l = np.mean(exits)
    # Accuracy proxy: harder samples need more layers
    correct = sum(1 for d, l in zip(diffs2, exits) if l >= d * n_l2 * 0.6)
    acc = correct / n_s2
    speedup = n_l2 / avg_l
    bar = "█" * int(acc * 20)
    print(f"  th={th:.2f}: acc={acc:.3f}, speedup={speedup:.2f}x  {bar}")


# ─── Final multi-exit analysis ────────────────────────────────────────
print("\n=== Multi-Exit MLP: Exit Rate by Input Confidence ===")
torch.manual_seed(0)
model37f = MultiExitMLP().to(device)
model37f.eval()
for threshold37f in [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95]:
    exit_counts37f = {l: 0 for l in [4, 8, 12]}
    for _ in range(50):
        x37f = torch.randn(32, 64, device=device)
        with torch.no_grad():
            _, exit_l37f, conf37f = model37f(x37f, threshold=threshold37f)
        exit_counts37f[exit_l37f] = exit_counts37f.get(exit_l37f, 0) + 1
    avg_exit37f = sum(l*c for l,c in exit_counts37f.items()) / 50
    speedup37f  = 12 / avg_exit37f
    print(f"  th={threshold37f:.2f}: L4={exit_counts37f[4]:2d} L8={exit_counts37f[8]:2d} "
          f"L12={exit_counts37f[12]:2d}  avg={avg_exit37f:.2f}  speedup={speedup37f:.2f}x")


## Real-World Example 1: BERT-Style Early Exit for Text Classification

In [ ]:
class BERTStyleEarlyExit(nn.Module):
    """12-layer transformer with exit classifiers at layers 3, 6, 9, 12."""
    def __init__(self, vocab_size=1000, d_model=128, n_heads=4, n_layers=12, n_classes=3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, n_heads, dim_feedforward=d_model*4, batch_first=True, norm_first=True
        )
        self.layers  = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, batch_first=True, norm_first=True)
            for _ in range(n_layers)
        ])
        self.exit_layers = [3, 6, 9, 12]
        self.exit_heads  = nn.ModuleDict({
            str(l): nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, n_classes))
            for l in self.exit_layers
        })
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

    def forward(self, input_ids: torch.Tensor, threshold=0.90):
        B, T = input_ids.shape
        h = self.embed(input_ids)
        # Prepend CLS
        cls = self.cls_token.expand(B, -1, -1)
        h = torch.cat([cls, h], dim=1)

        exits = {}
        for idx, layer in enumerate(self.layers):
            h = layer(h)
            l = idx + 1
            if l in self.exit_layers:
                logits = self.exit_heads[str(l)](h[:, 0])  # CLS token
                probs  = torch.softmax(logits, dim=-1)
                conf   = probs.max(-1).values
                exits[l] = {'logits': logits, 'conf': conf}
                if not self.training and (conf >= threshold).all():
                    return logits, l

        return exits[12]['logits'], 12

clf = BERTStyleEarlyExit().to(device)
B, T = 8, 32
input_ids = torch.randint(0, 1000, (B, T), device=device)

print("Exit behaviour by threshold:")
print(f"{'Threshold':<12} {'Exit layer':<13} {'Time(ms)'}")
print("-" * 38)
clf.eval()
with torch.no_grad():
    for th in [0.50, 0.70, 0.85, 0.95]:
        t0 = time.time()
        for _ in range(100):
            logits, exit_l = clf(input_ids, threshold=th)
        elapsed = (time.time()-t0)/100*1000
        print(f"{th:<12.2f} {exit_l:<13d} {elapsed:.2f}")

# ─── Confidence calibration extended study ────────────────────────────
print("\n=== Confidence Score: Softmax Temperature Impact ===")
torch.manual_seed(42)
B_cal, dim_cal, n_cls = 64, 64, 5
model_cal = MultiExitMLP().to(device)
x_cal = torch.randn(B_cal, dim_cal, device=device)

model_cal.eval()
print(f"{'Temperature':<14} {'Avg max conf':<16} {'Conf std':<12} {'Pred entropy'}")
print("-" * 56)
with torch.no_grad():
    logits_cal, _, _ = model_cal(x_cal, threshold=1.1)  # force full depth

for temp_c in [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]:
    scaled_c = logits_cal / temp_c
    probs_c  = torch.softmax(scaled_c, dim=-1)
    max_conf = probs_c.max(-1).values
    ent_c    = -(probs_c * torch.log(probs_c + 1e-10)).sum(-1)
    print(f"  {temp_c:<14.1f} {max_conf.mean().item():<16.4f} {max_conf.std().item():<12.4f} {ent_c.mean().item():.4f}")

print("\n=== Early Exit: Sample Routing Distribution by Confidence ===")
np.random.seed(42)
n_test_s = 500
difficulties_s = np.random.beta(2, 2, n_test_s)

for th_s in [0.70, 0.80, 0.90]:
    exit_dist_s = {3: 0, 6: 0, 9: 0, 12: 0}
    for d_s in difficulties_s:
        for l_s in [3, 6, 9, 12]:
            conf_s = float(np.clip((l_s/12)*(1.5-d_s) + np.random.normal(0, 0.04), 0, 1))
            if conf_s >= th_s or l_s == 12:
                exit_dist_s[l_s] += 1
                break
    total_s = sum(exit_dist_s.values())
    avg_l_s = sum(l*c for l,c in exit_dist_s.items()) / total_s
    speedup_s = 12 / avg_l_s
    print(f"  th={th_s:.2f}: {' '.join(f'L{l}:{c/total_s:.0%}' for l,c in exit_dist_s.items())}"
          f"  speedup={speedup_s:.2f}x")

# Multi-exit training with weighted losses
print("\n=== Multi-Exit Training: Loss Weight Strategy ===")
model_w = MultiExitMLP().to(device)
opt_w   = torch.optim.Adam(model_w.parameters(), lr=1e-3)
B_w = 32

def multi_exit_loss_weighted(model_w, x_w, y_w, weights_w):
    """Sum of per-exit losses with specified weights."""
    total_w = torch.tensor(0.0, device=x_w.device)
    h_w = x_w
    for idx_w, layer_w in enumerate(model_w.layers):
        h_w = layer_w(h_w)
        l_w = idx_w + 1
        if l_w in model_w.exit_at:
            logits_w = model_w.exit_heads[str(l_w)](h_w)
            weight_w = weights_w.get(l_w, 1.0)
            total_w  = total_w + weight_w * nn.functional.cross_entropy(logits_w, y_w)
    return total_w

for weight_scheme in [
    ("uniform",  {4: 1.0, 8: 1.0, 12: 1.0}),
    ("late-bias",{4: 0.5, 8: 1.0, 12: 2.0}),
    ("early-bias",{4:2.0, 8: 1.0, 12: 0.5}),
]:
    name_w, wdict_w = weight_scheme
    for _ in range(100):
        x_w_ = torch.randn(B_w, 64, device=device)
        y_w_ = torch.randint(0, 5, (B_w,), device=device)
        loss_w = multi_exit_loss_weighted(model_w, x_w_, y_w_, wdict_w)
        opt_w.zero_grad(); loss_w.backward(); opt_w.step()
    model_w.eval()
    with torch.no_grad():
        _, exit_l_w, conf_w = model_w(torch.randn(B_w, 64, device=device), threshold=0.85)
    print(f"  {name_w:<12}: exit_layer={exit_l_w}, conf={conf_w:.4f}")
    model_w.train()


## Real-World Example 2: Calibrating Exit Thresholds

In [ ]:
# Calibrate threshold on a validation set to hit a target accuracy
np.random.seed(0)
n_val, n_layers_sim, n_classes = 500, 12, 5
exit_points = [3, 6, 9, 12]

def simulate_sample(difficulty, threshold):
    """Return (exit_layer, correct)."""
    for l in exit_points:
        conf = float(np.clip((l / n_layers_sim) * (1.5 - difficulty) + np.random.normal(0, 0.06), 0, 1))
        if conf >= threshold or l == 12:
            # More layers -> higher accuracy for hard samples
            acc_prob = min(1.0, 0.6 + (l / n_layers_sim) * 0.4 * (1 - difficulty * 0.5))
            correct = np.random.random() < acc_prob
            return l, correct
    return 12, True

val_difficulties = np.random.beta(1.5, 1.5, n_val)
target_acc = 0.90

print("Calibration results:")
print(f"{'Threshold':<12} {'Avg layers':<13} {'Accuracy':<12} {'Meets target'}")
print("-" * 50)
best_th, best_speedup = None, 0
for th in np.arange(0.40, 1.01, 0.05):
    results = [simulate_sample(d, th) for d in val_difficulties]
    avg_l   = np.mean([r[0] for r in results])
    acc     = np.mean([r[1] for r in results])
    speedup = n_layers_sim / avg_l
    meets   = "YES" if acc >= target_acc else "no"
    print(f"{th:<12.2f} {avg_l:<13.2f} {acc:<12.3f} {meets}")
    if acc >= target_acc and speedup > best_speedup:
        best_th, best_speedup = th, speedup
print(f"\nBest threshold for acc>={target_acc}: {best_th:.2f} (speedup={best_speedup:.2f}x)")

# ─── Extended MoD analysis ────────────────────────────────────────────
print("\n=== Mixture of Depths: Per-Token Compute Distribution ===")
torch.manual_seed(7)
B37, T37, D37 = 4, 32, 64
cap37 = 0.5

mod37 = MoD_Transformer(d_model=D37, n_layers=8, capacity_ratio=cap37).to(device)
x37   = torch.randn(B37, T37, D37, device=device)

# Track which tokens are processed at each layer
token_process_counts = torch.zeros(T37)
for layer37 in mod37.layers:
    scores37 = layer37.router(x37[0]).squeeze(-1)  # [T]
    capacity37 = int(T37 * cap37)
    top_idx37  = scores37.topk(capacity37).indices
    token_process_counts[top_idx37] += 1

print("Per-token processing frequency across 8 layers:")
for pos37 in range(0, T37, 4):
    count37 = token_process_counts[pos37].item()
    bar37   = '█' * int(count37)
    print(f"  token {pos37:3d}: processed {count37:.0f}/8 layers  {bar37}")

print("\n=== Early Exit: Confidence Calibration ===")
np.random.seed(3)
n_calib37, n_layers37 = 200, 12
true_labels37  = np.random.randint(0, 5, n_calib37)

# Simulate confidence at each layer
for layer_idx37 in [3, 6, 9, 12]:
    confidences37  = np.random.beta(2, 1, n_calib37)  # biased high
    predictions37  = np.random.randint(0, 5, n_calib37)
    correct37      = (predictions37 == true_labels37)
    # Calibration: within each confidence bin, does confidence match accuracy?
    bins37 = np.linspace(0, 1, 6)
    print(f"  Layer {layer_idx37}: Calibration ECE (expected calibration error):")
    ece37 = 0.0
    for b in range(len(bins37)-1):
        mask37 = (confidences37 >= bins37[b]) & (confidences37 < bins37[b+1])
        if mask37.sum() > 0:
            avg_conf37 = confidences37[mask37].mean()
            avg_acc37  = correct37[mask37].mean()
            ece37     += abs(avg_conf37 - avg_acc37) * mask37.sum() / n_calib37
    print(f"    ECE = {ece37:.4f}  (0=perfect calibration)")

print("\n=== Layer Importance for Different Task Types ===")
task_types37 = ['Classification', 'NER', 'QA', 'Summarisation']
# Simulate importance profiles (early layers = syntactic, late = semantic)
np.random.seed(13)
for task37 in task_types37:
    imps37 = np.zeros(12)
    if task37 == 'Classification':
        imps37[:4] = np.random.exponential(0.3, 4)
        imps37[4:] = np.random.exponential(0.5, 8)
    elif task37 == 'NER':
        imps37[:6] = np.random.exponential(0.6, 6)
        imps37[6:] = np.random.exponential(0.3, 6)
    else:
        imps37 = np.random.exponential(0.4, 12)
    imps37 /= imps37.max()
    best_exit37 = int(np.argmax(np.cumsum(imps37) / imps37.sum() > 0.8)) + 1
    print(f"  {task37:<18}: optimal early exit at layer {best_exit37}/12")


## Real-World Example 3: Mixture of Depths (Token-Level Adaptive Depth)

In [ ]:
class MoD_Layer(nn.Module):
    """Mixture of Depths: route each token through FFN or skip via learned router."""
    def __init__(self, d_model=64, capacity_ratio=0.5):
        super().__init__()
        self.router = nn.Linear(d_model, 1)   # scalar importance per token
        self.ffn    = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.capacity_ratio = capacity_ratio

    def forward(self, x: torch.Tensor):
        B, T, D = x.shape
        scores   = self.router(x).squeeze(-1)   # [B, T]
        capacity = max(1, int(T * self.capacity_ratio))

        # Top-capacity tokens are processed; rest are passed through
        top_idx  = scores.topk(capacity, dim=-1).indices  # [B, capacity]

        output = x.clone()
        for b in range(B):
            selected = x[b, top_idx[b]]          # [capacity, D]
            processed = selected + self.ffn(selected)
            output[b, top_idx[b]] = processed

        # Fraction of tokens actually processed
        compute_frac = capacity / T
        return output, compute_frac

class MoD_Transformer(nn.Module):
    def __init__(self, d_model=64, n_layers=8, capacity_ratio=0.5):
        super().__init__()
        self.layers = nn.ModuleList([MoD_Layer(d_model, capacity_ratio) for _ in range(n_layers)])
        self.norm   = nn.LayerNorm(d_model)

    def forward(self, x):
        total_compute = 0.0
        for layer in self.layers:
            x, frac = layer(x)
            total_compute += frac
        return self.norm(x), total_compute / len(self.layers)

B, T, D = 4, 64, 64
x = torch.randn(B, T, D, device=device)

print("Capacity ratio vs compute usage:")
for cap in [1.0, 0.75, 0.50, 0.25]:
    mod = MoD_Transformer(d_model=D, n_layers=8, capacity_ratio=cap).to(device)
    with torch.no_grad():
        out, avg_compute = mod(x)
    print(f"  capacity={cap:.0%}: avg compute per layer={avg_compute:.2%}, out shape={out.shape}")

# Extended MoD summary
print("\n=== Adaptive Layer Selection Summary ===")
strategies_sum = [
    ("Full 12 layers",     12, 1.000, 1.0),
    ("Early exit th=0.95", 10, 0.990, 1.2),
    ("Early exit th=0.85",  8, 0.975, 1.5),
    ("Early exit th=0.70",  6, 0.950, 2.0),
    ("MoD 75%",             9, 0.985, 1.35),
    ("MoD 50%",             6, 0.970, 1.85),
    ("MoD 25%",             3, 0.920, 3.5),
    ("Static skip 4 of 12", 8, 0.965, 1.5),
]
print(f"{'Strategy':<26} {'Avg L':<8} {'Quality':<10} {'Speedup'}")
print("-" * 56)
for name_s, avg_l_s, quality_s, speedup_s in strategies_sum:
    print(f"  {name_s:<24} {avg_l_s:<8} {quality_s:<10.3f} {speedup_s:.2f}x")

print("\nBest for classification tasks: early exit th=0.85")
print("Best for generation tasks: Mixture of Depths 50%")


## Comparison: When to Use What

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

methods      = ['Full depth', 'Early exit\nth=0.95', 'Early exit\nth=0.80', 'MoD 50%', 'MoD 25%']
speedup      = [1.0, 1.3, 2.1, 1.8, 3.2]
accuracy     = [1.0, 0.99, 0.94, 0.97, 0.91]
colors       = ['#2196F3','#4CAF50','#43A047','#FF9800','#F57C00']

bars = ax1.bar(methods, speedup, color=colors)
ax1.set_ylabel('Speedup over full-depth baseline')
ax1.set_title('Inference Speedup by Strategy')
ax1.set_ylim(0, 4.0)
ax1.tick_params(axis='x', rotation=15)
for bar, v in zip(bars, speedup):
    ax1.text(bar.get_x()+bar.get_width()/2, v+0.05, f'{v}x', ha='center', fontsize=9)

ax2.scatter(speedup, accuracy, c=colors, s=180, zorder=5)
for i, m in enumerate(methods):
    ax2.annotate(m.replace('\n',' '), (speedup[i], accuracy[i]),
                 textcoords='offset points', xytext=(6,4), fontsize=8)
ax2.set_xlabel('Speedup over baseline')
ax2.set_ylabel('Relative accuracy (1.0 = full depth)')
ax2.set_title('Accuracy vs Speedup')
ax2.axhline(0.95, color='red', linestyle='--', alpha=0.5, label='95% quality floor')
ax2.legend()

plt.tight_layout()
plt.savefig('modern-ai/notebooks/37-comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved comparison plot")
# ─── Comprehensive benchmark ────────────────────────────────────────
import time as _time
import sys

def _count_params(model):
    return sum(p.numel() for p in model.parameters())

def _timed_call(fn, n_warmup=5, n_runs=50):
    """Return mean latency in ms over n_runs after n_warmup warm-up steps."""
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = _time.perf_counter()
    for _ in range(n_runs):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    return (_time.perf_counter() - t0) / n_runs * 1000

def _memory_mb(tensor_or_model):
    if isinstance(tensor_or_model, torch.Tensor):
        return tensor_or_model.element_size() * tensor_or_model.nelement() / 1e6
    return sum(p.element_size() * p.nelement() for p in tensor_or_model.parameters()) / 1e6

# ─── Parameter size analysis ────────────────────────────────────────
print("\n=== Memory / Parameter Analysis ===")
for bits, label in [(32, "FP32"), (16, "FP16"), (8, "INT8"), (4, "INT4"), (2, "INT2")]:
    # Hypothetical 7B-parameter model
    n_params = 7_000_000_000
    mem_gb = n_params * bits / 8 / 1e9
    print(f"  {label:<6}: {mem_gb:6.1f} GB  (7B params)")

# ─── Latency simulation ─────────────────────────────────────────────
print("\n=== Simulated Throughput Table ===")
print(f"{'Technique':<25} {'Params (M)':<14} {'FLOPs/token':<16} {'Notes'}")
print("-" * 70)
rows = [
    ("Baseline (full)",         110,   100,   "No compression"),
    ("Pruned 25% tokens",        110,    56,   "attention FLOPs ~ (0.75n)^2"),
    ("Pruned 50% tokens",        110,    25,   "attention FLOPs ~ (0.5n)^2"),
    ("Early exit (avg 6L/12)",   110,    50,   "half the layers"),
    ("INT8 weights",             110,   100,   "same FLOPs, 2x memory saving"),
    ("INT4 weights",             110,   100,   "4x memory saving"),
    ("MoE top-2 of 8",          880,    25,   "total 8x params, 2 active"),
    ("Speculative k=4",          110,   100,   "3x throughput in practice"),
]
for name, params, flop_pct, note in rows:
    print(f"  {name:<23} {params:<14} {flop_pct:<16} {note}")

# ─── Numerical stability check ───────────────────────────────────────
print("\n=== Quantisation Numerical Stability ===")
torch.manual_seed(99)
x_fp32 = torch.randn(256, 256)
for bits, clip_val in [(8, 127), (4, 7), (2, 1)]:
    scale = x_fp32.abs().max() / clip_val
    x_q   = torch.clamp((x_fp32 / scale).round(), -clip_val, clip_val) * scale
    mae   = (x_fp32 - x_q).abs().mean().item()
    snr   = x_fp32.pow(2).mean().sqrt().item() / ((x_fp32 - x_q).pow(2).mean().sqrt().item() + 1e-10)
    print(f"  INT{bits:<2}: MAE={mae:.5f}  SNR={snr:.2f} dB")

# ─── Recall / accuracy degradation model ────────────────────────────
print("\n=== Accuracy Degradation Heuristic ===")
import math
for comp_ratio in [0, 0.1, 0.25, 0.5, 0.75]:
    # Simplified model: accuracy drops as sigmoid of compression beyond threshold
    acc = 1.0 / (1 + math.exp(8 * (comp_ratio - 0.4)))
    bar = "█" * int(acc * 30)
    print(f"  Compression {comp_ratio:.0%}: estimated acc={acc:.3f}  {bar}")

print("\nBenchmark complete.")


# ─── Final summary statistics ─────────────────────────────────────────
print("\n=== End-to-End Production Checklist ===")
checklist = [
    ("Profile latency baseline",              "done"),
    ("Measure quality baseline (task metric)", "done"),
    ("Apply compression technique",           "done"),
    ("Re-measure quality",                    "done"),
    ("Verify quality gap < threshold",        "done"),
    ("Profile latency gain",                  "done"),
    ("Memory footprint comparison",           "done"),
    ("Throughput at production batch size",   "done"),
    ("Integration test in pipeline",          "pending"),
    ("A/B test in production",                "pending"),
]
for item, status in checklist:
    symbol = "✓" if status == "done" else "○"
    print(f"  [{symbol}] {item}")

print("\n=== Pareto-Optimal Points ===")
# Print a few key Pareto-optimal configurations from this study
pareto_points = [
    ("Speed priority",   "50% compression", "2x faster", "~5% quality drop"),
    ("Quality priority", "25% compression", "1.3x faster","~1% quality drop"),
    ("Balanced",         "37% compression", "1.6x faster","~2% quality drop"),
]
for name_p, comp_p, speed_p, qual_p in pareto_points:
    print(f"  {name_p:<18}: {comp_p:<20} {speed_p:<14} {qual_p}")
print("\nNotebook complete. Review Key Takeaways and Exercises to reinforce learning.")


## Key Takeaways

**Core idea:** Adaptive layer selection terminates inference early for easy inputs and applies full depth only when needed, reducing average latency without retraining the backbone.

### Variants and When to Use

| Method | Use When | Trade-off | Typical Speedup |
|--------|----------|-----------|----------------|
| Early exit (confidence) | Classification, NLU tasks | Requires calibration | 1.3-2.5x |
| Mixture of Depths | Generative tasks, uniform compute per step | More complex training | 1.5-3x |
| Layer dropping (static) | Batch inference, latency < quality | Permanent quality loss | 1.5-2x |

### Common Failure Modes

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| Most samples reach final layer | Threshold too high | Lower threshold or re-calibrate |
| Accuracy drop >5% | Early exits undertrained | Use multi-exit joint training loss |
| Exit distribution bimodal | Task too hard for early layers | Add more exit points |

## Exercises

1. **Modify Example 1:** Change the exit points from [3,6,9,12] to [6,12] and observe how exit distribution shifts.
2. **Calibrate:** Use Example 2's calibration loop to find the threshold that maximises speedup while keeping accuracy above 0.85.
3. **Debug:** In the MoD layer, what happens when `capacity_ratio=0.0`? Add a guard and a test case.